# WEIGHT1 밤샘 묶음 — 여지 확인 + M4 주변 가중치 + M4에서 출발하는 학습

이 노트북은 세 가지를 한 번에 돌립니다. 모두 **반복 사용한 날짜의 추가 개발 평가**이며, 판정 규칙은 실행 전에 `config/explore_night_v1.json`에 고정했습니다.

| 이름 | 무엇을 | 대략 크기 |
|---|---|---|
| N1 | 탐색 제어를 시험했던 6쌍에서 P0와 고정 M4를 다시 돌려, 탐색이 M4보다 정말 나았는지 같은 쌍으로 비교 | 12회 |
| N2 | M4(좌회전 가중치 2.0) 주변의 고정 가중치 5가지를 개발 16조건에서 비교 | 80회 |
| N3 | M4에서 출발하는 듀얼링 DQN을 학습 seed 3개 × 139에피소드 학습하고 42·139에피소드 모델을 개발 16조건에서 평가 | 학습 417에피소드 + 평가 96회 |

**순서**
1. 새 ZIP `FutureTraffic_NIGHT.zip`을 Drive의 `MyDrive/FutureTraffic`에 둡니다. 기존 폴더(`explore_weight_resume_v1` 등)는 건드리지 않습니다.
2. Colab에서 **런타임 → 모두 실행**을 누릅니다.
3. **처음 약 10분은 지켜봐 주세요.** 검사와 ‘스모크 테스트’(실제 SUMO로 에피소드 3개)가 끝나고 `SMOKE PASSED`가 보이면 그 뒤는 자동으로 이어지니 주무셔도 됩니다. 실패하면 그 자리에서 멈추고 이유를 보여 줍니다.
4. 끝나면 마지막 칸이 요약을 보여 줍니다. 결과는 Drive의 `explore_night_v1/summary.md`에도 저장됩니다.

연결이 끊기면 다시 **모두 실행**하면 이어서 합니다(학습은 마지막 에피소드부터, 평가는 끝난 실행을 건너뜀). 계산 장부는 이전 실험(63.5시간)에 이어서 합산하며, 이번 밤샘은 30작업시간, 전체 96시간 한도를 넘으면 스스로 멈춥니다. 맥에서는 아무것도 실행하지 않습니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE_ROOT / "FutureTraffic_NIGHT.zip"
SOURCE = DRIVE_ROOT / "explore_weight_v1"
PARENT = DRIVE_ROOT / "explore_weight_resume_v1"
OUT = DRIVE_ROOT / "explore_night_v1"
WORKSPACE = Path("/content/FutureTraffic_NIGHT")
WORK = Path("/content/night_work")
assert ARCHIVE.is_file(), f"새 NIGHT ZIP이 없습니다: {ARCHIVE}"
assert (SOURCE / "experiment.json").is_file(), f"원본 실험 폴더가 없습니다: {SOURCE}"
assert (PARENT / "budget.json").is_file() and (PARENT / "runs" / "development_references").is_dir(), f"이전 실험(Resume) 폴더가 없습니다: {PARENT}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS)
if CPUS < 4:
    print("경고: CPU가 4개 미만입니다. 학습 3개와 평가를 동시에 돌리면 오래 걸리고 작업시간이 늘어납니다.")
print("새 결과 폴더:", OUT)

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib
import json
import stat
import sys
import types
import zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "WEIGHT1_NIGHT", "WEIGHT1_NIGHT 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델은 Drive 원본에서 읽어야 합니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("weight1_night_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE}")

In [ ]:
import subprocess
import importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}
source_packages = json.loads((SOURCE / "experiment.json").read_text())["packages"]
assert all(source_packages.get(name) == version for name, version in EXPECTED.items()), "출발 실험의 패키지 버전이 이 묶음과 다릅니다."

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("출발 실험과 패키지 버전이 모두 같아 재설치를 건너뜁니다.")
else:
    print("출발 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 출발 실험과 다릅니다: {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
ENV = {**os.environ, "FOCUSED_RULESET": "pet1", "FOCUSED_NORTH_EXT": "240", "PYTHONUNBUFFERED": "1",
       "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"}
COMMON = ["--out", str(OUT), "--work", str(WORK), "--parent", str(PARENT)]

def run_live(args, check=True):
    """자식 프로세스의 출력을 실시간으로 이 칸에 보여 줍니다."""
    process = subprocess.Popen([sys.executable, "-u", *args], cwd=WORKSPACE, env=ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line.rstrip())
    code = process.wait()
    if check and code != 0:
        raise RuntimeError(f"명령이 실패했습니다(종료 코드 {code}): {' '.join(args[:2])}")
    return code

# 실제 SUMO 없이 하는 필수 검사 (기존 WEIGHT1 검사 + 이번 밤샘 검사)
for pattern in ("test_explore_weight*.py", "test_explore_night*.py"):
    run_live(["-m", "unittest", "discover", "-s", "tests", "-p", pattern])
print("검사 통과")

In [ ]:
# 등록(코드·설정·자료 지문 고정, 이전 실험 누적 사용량 기록) → 스모크 테스트(실제 SUMO 에피소드 3개, 약 5~10분)
run_live(["src/explore_night.py", "prepare", *COMMON])
fingerprint = json.loads((OUT / "experiment.json").read_text())["fingerprint"]
report_path = OUT / "smoke" / "report.json"
passed_before = False
if report_path.is_file():
    previous = json.loads(report_path.read_text())
    passed_before = bool(previous.get("passed")) and previous.get("fingerprint") == fingerprint
if passed_before:
    print("같은 코드로 이미 통과한 스모크 테스트가 있어 건너뜁니다.")
else:
    run_live(["src/explore_night.py", "smoke", *COMMON], check=False)
    report = json.loads(report_path.read_text()) if report_path.is_file() else {"passed": False, "problems": ["보고서가 없습니다"]}
    if not report.get("passed"):
        raise RuntimeError(f"스모크 테스트 실패 — 밤샘 실행을 시작하지 않습니다: {report.get('problems')}")
print("SMOKE PASSED — 이제 아래 칸이 자동으로 밤샘 실행을 이어 갑니다. 주무셔도 됩니다.")

In [ ]:
# 밤샘 실행: 학습 3개 + 점검(N1·N2)을 동시에 돌리고, 끝나면 학습 모델을 평가한 뒤 요약합니다.
import signal
import time
from IPython.display import clear_output, display, Markdown

LOGS = OUT / "logs"
LOGS.mkdir(parents=True, exist_ok=True)
SEEDS = (42, 43, 44)
MAX_STARTS = 4
PROCS, STARTS = {}, {}

def status(name):
    path = OUT / "status" / f"{name}.json"
    try:
        return json.loads(path.read_text())
    except (OSError, ValueError):
        return None

def live_pids(tokens):
    found = []
    for entry in Path("/proc").iterdir():
        if not entry.name.isdigit():
            continue
        try:
            parts = entry.joinpath("cmdline").read_bytes().rstrip(b"\0").split(b"\0")
        except OSError:
            continue
        parts = [p.decode("utf-8", "replace") for p in parts]
        if "explore_night.py" in " ".join(parts) and all(t in parts for t in tokens):
            found.append(int(entry.name))
    return found

def active(name, tokens):
    process = PROCS.get(name)
    return (process is not None and process.poll() is None) or bool(live_pids(tokens))

def launch(name, args, tokens):
    STARTS[name] = STARTS.get(name, 0) + 1
    stream = (LOGS / f"{name}.log").open("ab", buffering=0)
    PROCS[name] = subprocess.Popen([sys.executable, "-u", "src/explore_night.py", *args, *COMMON], cwd=WORKSPACE,
                                   stdout=stream, stderr=subprocess.STDOUT, env=ENV, start_new_session=True)
    print(time.strftime("%H:%M:%S"), f"시작: {name} (시도 {STARTS[name]}) PID {PROCS[name].pid}")

def trainer_done(seed):
    state = status(f"train_{seed}")
    return bool(state) and state["state"] in ("complete", "limit")

def group_done(name):
    state = status(name)
    return bool(state) and not state.get("stopped") and not state.get("failed")

def group_stopped(name):
    state = status(name)
    return bool(state) and bool(state.get("stopped"))

def episodes_of(seed):
    try:
        pointer = json.loads((OUT / "models" / f"seed_{seed}" / "latest.json").read_text())["checkpoint"]
        return int(Path(pointer).name.split("_")[1])
    except (OSError, ValueError, IndexError, KeyError):
        return 0

def count_runs(part):
    folder = OUT / "runs" / part
    return len(list(folder.glob("*/metrics.json"))) if folder.is_dir() else 0

def night_hours():
    try:
        return sum(json.loads(p.read_text())["seconds"] for p in (OUT / "ledger").glob("*.json") if p.name != "prior.json") / 3600
    except (OSError, ValueError):
        return 0.0

def tail(name, n=2):
    try:
        return (LOGS / f"{name}.log").read_text(errors="replace").strip().splitlines()[-n:]
    except OSError:
        return []

def stop_all():
    for name, process in PROCS.items():
        if process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGINT)
            except ProcessLookupError:
                pass
    time.sleep(20)
    for name, process in PROCS.items():
        if process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass

def dashboard(phase):
    clear_output(wait=True)
    print(time.strftime("%Y-%m-%d %H:%M:%S"), "| 단계:", phase, f"| 이번 밤샘 작업시간 {night_hours():.2f}h")
    for seed in SEEDS:
        state = status(f"train_{seed}")
        print(f"  학습 seed {seed}: {episodes_of(seed)}/139 에피소드, 상태 {state['state'] if state else '-'}")
    print(f"  점검: N1 {count_runs('n1')}/12, 재현 {count_runs('repro')}/4, N2 {count_runs('n2')}/80")
    print(f"  평가: N3 {count_runs('n3')}/{16 * 2 * len(SEEDS)}")
    for name in [f"train_{s}" for s in SEEDS] + ["checks", "evaluate"]:
        for line in tail(name):
            print(f"    [{name}] {line[:160]}")

phase = "학습·점검"
try:
    while True:
        if phase == "학습·점검":
            for seed in SEEDS:
                name, tokens = f"train_{seed}", ["train", str(seed)]
                if not trainer_done(seed) and not active(name, tokens) and STARTS.get(name, 0) < MAX_STARTS:
                    launch(name, ["src/explore_night.py", "train", "--seed", str(seed)][1:], tokens)
            if not group_done("checks") and not group_stopped("checks") and not active("checks", ["checks"]) and STARTS.get("checks", 0) < MAX_STARTS:
                launch("checks", ["src/explore_night.py", "checks", "--workers", "auto"][1:], ["checks"])
            idle = all((trainer_done(s) or not active(f"train_{s}", ["train", str(s)])) for s in SEEDS) and not active("checks", ["checks"])
            gave_up = all(trainer_done(s) or STARTS.get(f"train_{s}", 0) >= MAX_STARTS for s in SEEDS)
            if idle and gave_up and (group_done("checks") or group_stopped("checks") or STARTS.get("checks", 0) >= MAX_STARTS):
                phase = "평가"
        if phase == "평가":
            if not group_done("evaluate") and not group_stopped("evaluate") and not active("evaluate", ["evaluate"]):
                if STARTS.get("evaluate", 0) >= MAX_STARTS:
                    break
                launch("evaluate", ["src/explore_night.py", "evaluate", "--workers", str(CPUS)][1:], ["evaluate"])
            elif group_done("evaluate") or group_stopped("evaluate"):
                break
        dashboard(phase)
        time.sleep(60)
except KeyboardInterrupt:
    print("중단 요청: 실행 중인 작업을 정리합니다(학습은 마지막 에피소드부터 이어서 할 수 있습니다).")
    stop_all()
    raise
dashboard("끝")

In [ ]:
run_live(["src/explore_night.py", "summary", *COMMON], check=False)
summary_file = OUT / "summary.md"
if summary_file.is_file():
    display(Markdown(summary_file.read_text()))
print("원본 결과 폴더:", OUT)